In [ ]:
import boto3
import json
import logging
from botocore.exceptions import ClientError

# Initialize clients
iam = boto3.client("iam")
bedrock_agent = boto3.client("bedrock-agent")

# Setup logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


def get_policy_arn(policy_name):
    """Return the ARN of an IAM policy by name, if it exists."""
    paginator = iam.get_paginator("list_policies")
    for page in paginator.paginate(Scope="Local"):
        for pol in page["Policies"]:
            if pol["PolicyName"] == policy_name:
                return pol["Arn"]
    return None


def policy_attached(role_name, policy_arn):
    """Check if a policy is already attached to a role."""
    paginator = iam.get_paginator("list_attached_role_policies")
    for page in paginator.paginate(RoleName=role_name):
        for pol in page["AttachedPolicies"]:
            if pol["PolicyArn"] == policy_arn:
                return True
    return False


def associate_kb_with_agent(agent_id, kb_id, agent_role_name, region, account_id):
    # 1. Associate KB with Agent
    try:
        resp = bedrock_agent.associate_agent_knowledge_base(
            agentId=agent_id,
            knowledgeBaseId=kb_id
        )
        logger.info(f"✅ Associated KB {kb_id} with Agent {agent_id}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            logger.warning(f"⚠️ KB {kb_id} is already associated with Agent {agent_id}")
            resp = {"status": "already-associated"}
        else:
            raise

    # 2. Ensure IAM policy exists
    kb_policy_name = f"{agent_id}-kb-{kb_id}"
    kb_policy_arn = get_policy_arn(kb_policy_name)

    if not kb_policy_arn:
        kb_policy_doc = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "QueryKB",
                    "Effect": "Allow",
                    "Action": [
                        "bedrock:Retrieve",
                        "bedrock:RetrieveAndGenerate"
                    ],
                    "Resource": [
                        f"arn:aws:bedrock:{region}:{account_id}:knowledge-base/{kb_id}"
                    ]
                }
            ]
        }
        try:
            policy_resp = iam.create_policy(
                PolicyName=kb_policy_name,
                PolicyDocument=json.dumps(kb_policy_doc)
            )
            kb_policy_arn = policy_resp["Policy"]["Arn"]
            logger.info(f"✅ Created new policy {kb_policy_name}")
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityAlreadyExists":
                logger.warning(f"⚠️ Policy {kb_policy_name} exists, fetching ARN")
                kb_policy_arn = get_policy_arn(kb_policy_name)
            else:
                raise

    # 3. Attach policy to agent’s role (if not already attached)
    if kb_policy_arn and not policy_attached(agent_role_name, kb_policy_arn):
        iam.attach_role_policy(
            RoleName=agent_role_name,
            PolicyArn=kb_policy_arn
        )
        logger.info(f"✅ Attached KB policy {kb_policy_name} to {agent_role_name}")
    else:
        logger.info(f"ℹ️ Policy {kb_policy_name} already attached to {agent_role_name}")

    return resp


if __name__ == "__main__":
    # Replace with your values
    KB_ID = "REPLACE_WITH_YOUR_KB_ID"
    AGENT_ID = "REPLACE_WITH_YOUR_AGENT_ID"
    AGENT_ROLE_NAME = "AgentExecRole"   # must match role created in your agent script
    REGION = "us-east-1"
    ACCOUNT_ID = "123456789012"

    associate_kb_with_agent(AGENT_ID, KB_ID, AGENT_ROLE_NAME, REGION, ACCOUNT_ID)
    print(f"[OK] Linked KB {KB_ID} with Agent {AGENT_ID} and ensured IAM policy attachment")
